<a href="https://colab.research.google.com/github/tasiyaa/Surface/blob/main/intership_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NEU-DET Steel Surface Defect Detection

#  0. Installations

In [1]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'albumentations', 'timm', '-q'], check=False)

import math, copy, random
import numpy as np
import pandas as pd
import torch
import torchvision
import xml.etree.ElementTree as ET
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms.functional as TF
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.faster_rcnn import FasterRCNN, FastRCNNPredictor, TwoMLPHead
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import nms, MultiScaleRoIAlign
import torch.nn as nn
import torch.nn.functional as torch_F
from collections import Counter

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

if torch.cuda.is_available():
    torch.backends.cudnn.enabled   = False
    torch.backends.cudnn.benchmark = False
    DEVICE = torch.device('cuda')
    name   = torch.cuda.get_device_name(0)
    cap    = torch.cuda.get_device_capability(0)
    vram   = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  |  compute cap: {cap}  |  VRAM: {vram} GB')
    print(f'torch: {torch.__version__}  |  torchvision: {torchvision.__version__}')
else:
    DEVICE = torch.device('cpu')
    print('CPU mode')


GPU : Tesla T4  |  compute cap: (7, 5)  |  VRAM: 15.6 GB
torch: 2.11.0+cu128  |  torchvision: 0.26.0+cu128


# 1. Configuration

In [2]:
from google.colab import files
uploaded = files.upload()  # select the zip you downloaded

!unzip teep-internship-program-hucenrotia-lab.zip -d /content/dataset

Saving teep-internship-program-hucenrotia-lab.zip to teep-internship-program-hucenrotia-lab (1).zip
Archive:  teep-internship-program-hucenrotia-lab.zip
replace /content/dataset/Dataset/test/crazing_10dh18dy69jj.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/dataset/Dataset/test/crazing_10dh18dy69jj.jpg  
  inflating: /content/dataset/Dataset/test/crazing_17ej7vl96bai.jpg  
  inflating: /content/dataset/Dataset/test/crazing_1l6jrbqptp9x.jpg  
  inflating: /content/dataset/Dataset/test/crazing_1olaertr3kk3.jpg  
  inflating: /content/dataset/Dataset/test/crazing_1oskoqxf9ewd.jpg  
  inflating: /content/dataset/Dataset/test/crazing_2n9rfdh9adg4.jpg  
  inflating: /content/dataset/Dataset/test/crazing_3kyz2t5jhl4d.jpg  
  inflating: /content/dataset/Dataset/test/crazing_3zdq65hlyaa2.jpg  
  inflating: /content/dataset/Dataset/test/crazing_47ohkxigp0np.jpg  
  inflating: /content/dataset/Dataset/test/crazing_4azx7p0enj0d.jpg  
  inflating: /content/dataset/Dataset/test/

In [3]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR      = '/content/drive/MyDrive/neu_det_checkpoints'
CHECKPOINT_PATH = OUTPUT_DIR + '/checkpoint_best.pth'
EMA_PATH        = OUTPUT_DIR + '/ema_model.pth'
os.makedirs(OUTPUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
DATASET_ROOT   = '/content/dataset/Dataset'
TRAIN_IMG_ROOT = DATASET_ROOT + '/train_images'
TRAIN_ANN_ROOT = DATASET_ROOT + '/train_annotations'
VAL_IMG_ROOT   = '/content/dataset/Dataset/test'

NUM_CLASSES    = 7
EPOCHS         = 2
BATCH_SIZE     = 4       # 4 is safe on T4; gradient accum ×3 = effective 12
ACCUM_STEPS    = 3
LEARNING_RATE  = 1.5e-4
WEIGHT_DECAY   = 1e-4
CONF_THRESHOLD = 0.25    # lower = catch more at inference (TTA + NMS cleans up FPs)
NMS_THRESHOLD  = 0.45
USE_TTA        = True
INFER_ONLY     = False
IMG_SIZE       = 800     # larger image = better on small defects
SEED           = 42





SUBMISSION_PATH = OUTPUT_DIR + '/submission.csv'
VIZ_DIR         = OUTPUT_DIR + '/output_viz'
os.makedirs(VIZ_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

LABEL_MAP = {
    'crazing': 1, 'inclusion': 2, 'patches': 3,
    'pitted_surface': 4, 'rolled-in_scale': 5, 'scratches': 6,
}
REV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES   = ['bg'] + list(LABEL_MAP.keys())

for name, path in [('train_images', TRAIN_IMG_ROOT),
                   ('annotations',  TRAIN_ANN_ROOT),
                   ('test',         VAL_IMG_ROOT)]:
    ok = os.path.exists(path)
    n  = len(os.listdir(path)) if ok else 0
    print(f"{'OK' if ok else 'MISSING':6s}  {name:20s}  ({n} files)  {path}")

OK      train_images          (1440 files)  /content/dataset/Dataset/train_images
OK      annotations           (1440 files)  /content/dataset/Dataset/train_annotations
OK      test                  (360 files)  /content/dataset/Dataset/test


# 2. Augmentations

In [5]:
def get_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.4),
        A.RandomRotate90(p=0.4),
        A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.25,
                           rotate_limit=20, p=0.6, border_mode=0),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20),
            A.RandomGamma(gamma_limit=(70, 130)),
        ], p=0.7),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 60)),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1)),
        ], p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=3),
        ], p=0.3),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
        A.Sharpen(alpha=(0.2, 0.6), lightness=(0.5, 1.0), p=0.4),
        A.CoarseDropout(max_holes=8, max_height=40, max_width=40, fill_value=0, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'],
                                min_area=16, min_visibility=0.3))

def get_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'],
                                min_area=1, min_visibility=0.1))

# ── Mosaic augmentation (4-image mosaic like YOLOv5) ──────────────────────────
def mosaic_sample(dataset, idx, img_size=IMG_SIZE):
    # Combine 4 images into one mosaic for richer context.
    indices  = [idx] + random.sample(range(len(dataset)), 3)
    mosaic   = np.zeros((img_size * 2, img_size * 2, 3), dtype=np.uint8)
    all_boxes, all_labels = [], []
    positions = [(0, 0), (img_size, 0), (0, img_size), (img_size, img_size)]

    base_tf = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'],
                                min_area=1, min_visibility=0.1))

    for (ox, oy), i in zip(positions, indices):
        img_name = dataset.imgs[i]
        img      = np.array(Image.open(os.path.join(dataset.img_dir, img_name)).convert('RGB'))
        ann_path = os.path.join(dataset.ann_dir, img_name.replace('.jpg', '.xml'))
        boxes, labels = [], []
        for obj in ET.parse(ann_path).getroot().findall('object'):
            lbl  = obj.find('name').text.strip()
            bbox = obj.find('bndbox')
            xmin = int(float(bbox.find('xmin').text)); ymin = int(float(bbox.find('ymin').text))
            xmax = int(float(bbox.find('xmax').text)); ymax = int(float(bbox.find('ymax').text))
            h, w = img.shape[:2]
            xmin, xmax = max(0, xmin), min(w, xmax)
            ymin, ymax = max(0, ymin), min(h, ymax)
            if xmax > xmin + 2 and ymax > ymin + 2:
                boxes.append([xmin, ymin, xmax, ymax]); labels.append(LABEL_MAP[lbl])
        try:
            aug    = base_tf(image=img, bboxes=boxes, class_labels=labels)
            img    = aug['image']; boxes = list(aug['bboxes']); labels = list(aug['class_labels'])
        except: pass
        mosaic[oy:oy+img_size, ox:ox+img_size] = img
        for (x1,y1,x2,y2), lbl in zip(boxes, labels):
            all_boxes.append([x1+ox, y1+oy, x2+ox, y2+oy]); all_labels.append(lbl)

    # Crop back to img_size from centre
    cx, cy = img_size, img_size
    half   = img_size // 2
    mosaic = mosaic[cy-half:cy+half, cx-half:cx+half]
    final_boxes, final_labels = [], []
    for (x1,y1,x2,y2), lbl in zip(all_boxes, all_labels):
        nx1 = max(0, x1-(cx-half)); ny1 = max(0, y1-(cy-half))
        nx2 = min(img_size, x2-(cx-half)); ny2 = min(img_size, y2-(cy-half))
        if nx2 > nx1+4 and ny2 > ny1+4:
            final_boxes.append([nx1,ny1,nx2,ny2]); final_labels.append(lbl)
    return mosaic, final_boxes, final_labels

print('Augmentations + Mosaic ready')

Augmentations + Mosaic ready


# 3. Dataset

In [6]:
class NEUDETDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None, use_mosaic=False):
        self.img_dir     = img_dir
        self.ann_dir     = ann_dir
        self.transforms  = transforms
        self.use_mosaic  = use_mosaic
        self.imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')])
        self.all_labels = []
        for fname in self.imgs:
            ann  = os.path.join(ann_dir, fname.replace('.jpg', '.xml'))
            lbls = [LABEL_MAP[obj.find('name').text.strip()]
                    for obj in ET.parse(ann).getroot().findall('object')]
            self.all_labels.append(lbls[0] if lbls else 0)

    def _load_one(self, idx):
        img_name = self.imgs[idx]
        image    = np.array(Image.open(os.path.join(self.img_dir, img_name)).convert('RGB'))
        ann_path = os.path.join(self.ann_dir, img_name.replace('.jpg', '.xml'))
        boxes, labels = [], []
        for obj in ET.parse(ann_path).getroot().findall('object'):
            lbl  = obj.find('name').text.strip()
            bbox = obj.find('bndbox')
            xmin = int(float(bbox.find('xmin').text)); ymin = int(float(bbox.find('ymin').text))
            xmax = int(float(bbox.find('xmax').text)); ymax = int(float(bbox.find('ymax').text))
            h, w = image.shape[:2]
            xmin, xmax = max(0, xmin), min(w, xmax)
            ymin, ymax = max(0, ymin), min(h, ymax)
            if xmax > xmin + 2 and ymax > ymin + 2:
                boxes.append([xmin, ymin, xmax, ymax]); labels.append(LABEL_MAP[lbl])
        return image, boxes, labels

    def __getitem__(self, idx):
        # 30% chance of mosaic during training
        if self.use_mosaic and random.random() < 0.30 and len(self.imgs) >= 4:
            image, boxes, labels = mosaic_sample(self, idx, IMG_SIZE)
        else:
            image, boxes, labels = self._load_one(idx)

        if self.transforms and boxes:
            try:
                aug    = self.transforms(image=image, bboxes=boxes, class_labels=labels)
                image  = aug['image']; boxes = list(aug['bboxes']); labels = list(aug['class_labels'])
            except:
                image = torch.from_numpy(image.transpose(2,0,1)).float()/255.0
        elif self.transforms:
            aug   = self.transforms(image=image, bboxes=[], class_labels=[])
            image = aug['image']
        else:
            image = torch.from_numpy(image.transpose(2,0,1)).float()/255.0

        boxes  = (torch.as_tensor(boxes,  dtype=torch.float32)
                  if boxes else torch.zeros((0,4), dtype=torch.float32))
        labels = (torch.as_tensor(labels, dtype=torch.int64)
                  if labels else torch.zeros((0,),  dtype=torch.int64))
        area   = ((boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0])
                  if len(boxes) else torch.zeros(0, dtype=torch.float32))
        target = {'boxes': boxes, 'labels': labels,
                  'image_id': torch.tensor([idx], dtype=torch.int64),
                  'area': area, 'iscrowd': torch.zeros(len(labels), dtype=torch.int64)}
        return image, target

    def __len__(self): return len(self.imgs)


def get_balanced_sampler(dataset):
    lc = np.maximum(np.bincount(dataset.all_labels, minlength=NUM_CLASSES), 1)
    w  = 1.0 / lc
    sw = torch.tensor([w[l] for l in dataset.all_labels], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw))

def collate_fn(batch): return tuple(zip(*batch))

_ds = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT)
print(f'Train samples: {len(_ds)}')
dist = Counter(_ds.all_labels)
print('Class distribution:')
for cls_id, count in sorted(dist.items()):
    print(f'  {REV_LABEL_MAP.get(cls_id,"background"):20s}: {count}')

Train samples: 1440
Class distribution:
  crazing             : 240
  inclusion           : 240
  patches             : 240
  pitted_surface      : 240
  rolled-in_scale     : 240
  scratches           : 240


# 4. Model

In [7]:
# ── CBAM Channel + Spatial Attention ─────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_ch, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Flatten(), nn.Linear(in_ch, in_ch//ratio, bias=False),
            nn.ReLU(), nn.Linear(in_ch//ratio, in_ch, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.max(x))).view(x.size(0),-1,1,1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        attn = torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True)[0]], dim=1)
        return x * self.sig(self.conv(attn))

class CBAM(nn.Module):
    def __init__(self, in_ch, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(in_ch, ratio)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

# ── Enhanced ROI Box Head with CBAM + Dropout ─────────────────────────────────
class AttentionROIHead(nn.Module):
    # Drop-in replacement for TwoMLPHead with CBAM attention before flattening.
    def __init__(self, in_channels, representation_size, roi_output_size=7):
        super().__init__()
        self.cbam   = CBAM(in_channels)
        self.fc6    = nn.Linear(in_channels * roi_output_size * roi_output_size, representation_size)
        self.fc7    = nn.Linear(representation_size, representation_size)
        self.drop   = nn.Dropout(p=0.3)
        self.relu   = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.cbam(x)
        x = x.flatten(start_dim=1)
        x = self.relu(self.fc6(x))
        x = self.drop(x)
        x = self.relu(self.fc7(x))
        return x

def get_model(num_classes=NUM_CLASSES):
    # ── Build backbone ────────────────────────────────────────────────────────
    try:
        backbone = resnet_fpn_backbone(
            backbone_name='resnet101',
            weights='ResNet101_Weights.IMAGENET1K_V1',
            trainable_layers=5)
        print('Backbone: ResNet-101 FPN (ImageNet pretrained, fully unfrozen)')
    except Exception as e:
        print(f'ResNet-101 failed ({e.__class__.__name__}), falling back to ResNet-50...')
        backbone = resnet_fpn_backbone(
            backbone_name='resnet50',
            weights='ResNet50_Weights.IMAGENET1K_V2',
            trainable_layers=5)
        print('Backbone: ResNet-50 FPN (ImageNet V2 pretrained, fully unfrozen)')

    # ── Anchors tuned for NEU-DET defect sizes ────────────────────────────────
    # NEU-DET defects range from ~10px to ~180px on 200×200 images.
    # After resize to 800px they become ~40px–720px, so small anchors matter.
    anchor_gen = AnchorGenerator(
        sizes=((16,32), (32,64), (64,128), (128,256), (256,512)),
        aspect_ratios=((0.25, 0.5, 1.0, 2.0, 4.0),) * 5)

    # ── ROI pooling ───────────────────────────────────────────────────────────
    roi_pooler = MultiScaleRoIAlign(
        featmap_names=['0','1','2','3'],
        output_size=7, sampling_ratio=2)

    # ── Build Faster R-CNN ────────────────────────────────────────────────────
    model = FasterRCNN(
        backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_gen,
        box_roi_pool=roi_pooler,
        box_detections_per_img=300,
        rpn_nms_thresh=0.7,
        box_nms_thresh=NMS_THRESHOLD,
        box_score_thresh=CONF_THRESHOLD,
        rpn_pre_nms_top_n_train=4000,  rpn_post_nms_top_n_train=2000,
        rpn_pre_nms_top_n_test=2000,   rpn_post_nms_top_n_test=1000,
        min_size=800, max_size=1333)

    # ── Swap in AttentionROIHead ──────────────────────────────────────────────
    in_channels       = model.backbone.out_channels  # 256 for FPN
    representation_sz = 1024
    model.roi_heads.box_head = AttentionROIHead(in_channels, representation_sz, roi_output_size=7)
    model.roi_heads.box_predictor = FastRCNNPredictor(representation_sz, num_classes)

    # ── ROI sampling: more positives ──────────────────────────────────────────
    model.roi_heads.batch_size_per_image = 256   # was 512
    model.roi_heads.positive_fraction    = 0.5   # was 0.25  →  4× more positive signal

    # ── RPN: easier foreground matching for small defects ────────────────────
    model.rpn.fg_iou_thresh        = 0.6   # was 0.7
    model.rpn.bg_iou_thresh        = 0.3
    model.rpn.batch_size_per_image = 256
    model.rpn.positive_fraction    = 0.6

    return model.to(DEVICE)

model = get_model()
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Backbone: ResNet-101 FPN (ImageNet pretrained, fully unfrozen)
Trainable params: 60,281,975


#  5. EMA + scheduler + checkpoint

In [8]:
class ModelEMA:
    def __init__(self, model, decay=0.99995):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.ema.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)

def warmup_cosine_restarts(optimizer, warmup_epochs, total_epochs,
                            steps_per_epoch, n_restarts=3):
    # Cosine annealing with warm restarts after warmup.
    warmup_steps  = warmup_epochs * steps_per_epoch
    cycle_steps   = (total_epochs * steps_per_epoch - warmup_steps) // n_restarts

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        s        = step - warmup_steps
        cycle    = s % max(1, cycle_steps)
        progress = cycle / max(1, cycle_steps)
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))  # min 5% of LR

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def save_checkpoint(state, path):
    torch.save(state, path)

print('EMA (decay=0.99995) / cosine-restarts scheduler / checkpoint helpers ready')

EMA (decay=0.99995) / cosine-restarts scheduler / checkpoint helpers ready


#  6. mAP evaluation

In [9]:

def compute_map(model, val_loader, iou_threshold=0.5):
    model.eval()
    all_preds  = {i: [] for i in range(1, NUM_CLASSES)}
    all_gts    = {i: [] for i in range(1, NUM_CLASSES)}
    img_offset = 0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc='Evaluating', leave=False):
            images  = [img.to(DEVICE) for img in images]
            outputs = model(images)
            for i, (out, tgt) in enumerate(zip(outputs, targets)):
                gt_boxes  = tgt['boxes'].cpu()
                gt_labels = tgt['labels'].cpu()
                pd_boxes  = out['boxes'].cpu()
                pd_scores = out['scores'].cpu()
                pd_labels = out['labels'].cpu()
                img_id    = img_offset + i
                for cls in range(1, NUM_CLASSES):
                    gt_mask = gt_labels == cls
                    pd_mask = pd_labels == cls
                    all_gts[cls].append({'img_id': img_id, 'boxes': gt_boxes[gt_mask],
                                         'matched': torch.zeros(gt_mask.sum(), dtype=torch.bool)})
                    scores = pd_scores[pd_mask]; boxes = pd_boxes[pd_mask]
                    order  = scores.argsort(descending=True)
                    all_preds[cls].append({'img_id': img_id, 'scores': scores[order], 'boxes': boxes[order]})
            img_offset += len(images)

    aps, per_class = [], {}
    for cls in range(1, NUM_CLASSES):
        preds_flat = [(p['img_id'], s.item(), b)
                      for p in all_preds[cls]
                      for s, b in zip(p['scores'], p['boxes'])]
        preds_flat.sort(key=lambda x: -x[1])
        gt_by_img = {g['img_id']: g for g in all_gts[cls]}
        tp  = np.zeros(len(preds_flat))
        fp  = np.zeros(len(preds_flat))
        n_gt = sum(len(g['boxes']) for g in all_gts[cls])
        for k, (img_id, score, box) in enumerate(preds_flat):
            gt = gt_by_img.get(img_id)
            if gt is None or len(gt['boxes']) == 0: fp[k] = 1; continue
            ious = torchvision.ops.box_iou(box.unsqueeze(0), gt['boxes'])[0]
            best_iou, best_j = (ious.max(0) if len(ious) else (torch.tensor(0.), 0))
            if best_iou.item() >= iou_threshold and not gt['matched'][best_j]:
                tp[k] = 1; gt['matched'][best_j] = True
            else: fp[k] = 1
        cum_tp    = np.cumsum(tp); cum_fp = np.cumsum(fp)
        recall    = np.concatenate([[0], cum_tp / max(n_gt, 1), [1]])
        precision = np.concatenate([[1], cum_tp / (cum_tp + cum_fp + 1e-9), [0]])
        for j in range(len(precision) - 2, -1, -1):
            precision[j] = max(precision[j], precision[j + 1])
        ap = float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))
        aps.append(ap); per_class[REV_LABEL_MAP[cls]] = round(ap, 4)
    return float(np.mean(aps)) if aps else 0.0, per_class

print('mAP helper ready')

mAP helper ready


#  7. Training loop

In [10]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    # ── Separate param groups with different LRs ──────────────────────────────
    # backbone: very low LR (already well pretrained)
    # FPN + RPN + ROI head: full LR
    backbone_params = [p for n,p in model.named_parameters()
                       if 'backbone' in n and p.requires_grad]
    fpn_params      = [p for n,p in model.named_parameters()
                       if 'backbone' not in n and p.requires_grad]

    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LEARNING_RATE * 0.05},
        {'params': fpn_params,      'lr': LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = warmup_cosine_restarts(
        optimizer, warmup_epochs=5, total_epochs=epochs,
        steps_per_epoch=len(train_loader), n_restarts=3)

    ema      = ModelEMA(model, decay=0.99995)
    best_map = 0.0
    best_loss= 1e9
    history  = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        loss_comp  = {'cls':0., 'box':0., 'rpn_cls':0., 'rpn_box':0.}
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1:3d}/{epochs}')
        for step, (images, targets) in enumerate(pbar):
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]

            loss_dict = model(images, targets)

            # Weight classification loss higher — it's the accuracy bottleneck
            loss = (loss_dict.get('loss_classifier', torch.tensor(0.)) * 1.5 +
                    loss_dict.get('loss_box_reg', torch.tensor(0.))    * 1.0 +
                    loss_dict.get('loss_objectness', torch.tensor(0.)) * 1.0 +
                    loss_dict.get('loss_rpn_box_reg', torch.tensor(0.))* 0.8)
            loss = loss / ACCUM_STEPS

            for k,v in loss_dict.items():
                short = k.replace('loss_','')
                if short in loss_comp: loss_comp[short] += v.item()

            loss.backward()

            if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(backbone_params+fpn_params, max_norm=3.0)
                optimizer.step(); scheduler.step(); ema.update(model)
                optimizer.zero_grad()

            epoch_loss += loss.item() * ACCUM_STEPS
            pbar.set_postfix(loss=f'{loss.item()*ACCUM_STEPS:.3f}',
                             lr=f'{scheduler.get_last_lr()[-1]:.1e}')

        avg_loss = epoch_loss / len(train_loader)
        row = {'epoch': epoch+1, 'loss': avg_loss, 'lr': scheduler.get_last_lr()[-1]}

        map_score = 0.0
        if val_loader and (epoch+1) % 5 == 0:
            map_score, per_cls = compute_map(ema.ema, val_loader)
            model.train(); row['map'] = map_score
            print(f'  per-class AP: {per_cls}')

        history.append(row)
        n = len(train_loader)
        print(f'  loss={avg_loss:.4f}  '
              f'cls={loss_comp["cls"]/n:.3f}  '
              f'box={loss_comp["box"]/n:.3f}  '
              f'rpn={loss_comp["rpn_cls"]/n:.3f}  '
              f'lr={scheduler.get_last_lr()[-1]:.2e}'
              + (f'  mAP={map_score:.4f}' if map_score else ''))

        # Save on improved mAP OR improved loss (whichever tracked)
        save = ((map_score > best_map) if map_score
                else (avg_loss < best_loss))
        if save:
            if map_score: best_map = map_score
            if avg_loss < best_loss: best_loss = avg_loss
            save_checkpoint({'model': model.state_dict(), 'epoch': epoch,
                             'best_map': best_map, 'best_loss': best_loss}, CHECKPOINT_PATH)
            save_checkpoint(ema.ema.state_dict(), EMA_PATH)
            print('  ✓ Saved best checkpoint')

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR+'/train_history.csv', index=False)

    fig, axes = plt.subplots(1, 2 if 'map' in hist_df.columns else 1, figsize=(14,4))
    if not isinstance(axes, np.ndarray): axes=[axes]
    axes[0].plot(hist_df['epoch'], hist_df['loss'], marker='o', ms=2)
    axes[0].axhline(0.2, color='red', linestyle='--', alpha=0.5, label='target 0.2')
    axes[0].legend(); axes[0].set(xlabel='Epoch', ylabel='Loss', title='Training Loss')
    axes[0].grid(alpha=0.3)
    if 'map' in hist_df.columns:
        mdf = hist_df.dropna(subset=['map'])
        axes[1].plot(mdf['epoch'], mdf['map'], marker='s', ms=4, color='green')
        axes[1].axhline(0.80, color='red', linestyle='--', alpha=0.5, label='target 80%')
        axes[1].legend(); axes[1].set(xlabel='Epoch', ylabel='mAP@0.5', title='Validation mAP')
        axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR+'/training_curves.png', dpi=120); plt.show()
    return ema.ema

print(f'train_model ready  (grad_accum=×{ACCUM_STEPS}, effective_batch={BATCH_SIZE*ACCUM_STEPS})')

train_model ready  (grad_accum=×3, effective_batch=12)


# 8. TTA predict

In [11]:
@torch.no_grad()
def tta_predict(model, image_tensor):
    # 5-view TTA: original + hflip + vflip + rot90 + rot270
    def run(t):
        out = model(t)[0]
        return {k: v.cpu() for k,v in out.items()}

    results = []

    # Original
    results.append(run(image_tensor))

    # H-flip
    hf = TF.hflip(image_tensor[0]).unsqueeze(0)
    o  = run(hf); w = image_tensor.shape[-1]
    b  = o['boxes'].clone(); b[:,[0,2]] = w - b[:,[2,0]]
    results.append({'boxes':b,'scores':o['scores'],'labels':o['labels']})

    # V-flip
    vf = TF.vflip(image_tensor[0]).unsqueeze(0)
    o  = run(vf); h = image_tensor.shape[-2]
    b  = o['boxes'].clone(); b[:,[1,3]] = h - b[:,[3,1]]
    results.append({'boxes':b,'scores':o['scores'],'labels':o['labels']})

    # Rotate 90°
    r90 = torch.rot90(image_tensor[0], 1, [1,2]).unsqueeze(0)
    o   = run(r90); H,W = image_tensor.shape[-2], image_tensor.shape[-1]
    b   = o['boxes'].clone()
    # rot90 mapping: (x,y) -> (y, W-x)
    new_b = b.clone()
    new_b[:,0] = b[:,1]; new_b[:,1] = W - b[:,2]
    new_b[:,2] = b[:,3]; new_b[:,3] = W - b[:,0]
    results.append({'boxes':new_b,'scores':o['scores'],'labels':o['labels']})

    # Rotate 270°
    r270 = torch.rot90(image_tensor[0], 3, [1,2]).unsqueeze(0)
    o    = run(r270)
    b    = o['boxes'].clone()
    new_b = b.clone()
    new_b[:,0] = H - b[:,3]; new_b[:,1] = b[:,0]
    new_b[:,2] = H - b[:,1]; new_b[:,3] = b[:,2]
    results.append({'boxes':new_b,'scores':o['scores'],'labels':o['labels']})

    all_boxes  = torch.cat([r['boxes']  for r in results])
    all_scores = torch.cat([r['scores'] for r in results])
    all_labels = torch.cat([r['labels'] for r in results])
    if len(all_boxes) == 0: return results[0]

    fb, fs, fl = [], [], []
    for cls in all_labels.unique():
        mask = all_labels == cls
        keep = nms(all_boxes[mask], all_scores[mask], iou_threshold=0.45)
        fb.append(all_boxes[mask][keep])
        fs.append(all_scores[mask][keep])
        fl.append(all_labels[mask][keep])
    return {'boxes':torch.cat(fb),'scores':torch.cat(fs),'labels':torch.cat(fl)}

print('TTA ready  (5-view: orig + hflip + vflip + rot90 + rot270)')

TTA ready  (5-view: orig + hflip + vflip + rot90 + rot270)


# 9. Predict & export

In [12]:

def predict_and_export(model, val_dir, save_csv_path,
                        save_img_dir=VIZ_DIR, use_tta=True):
    os.makedirs(save_img_dir, exist_ok=True)
    model.eval()
    rows       = []
    val_images = sorted([f for f in os.listdir(val_dir) if f.lower().endswith('.jpg')])
    val_tf     = get_val_transforms()

    for img_file in tqdm(val_images, desc='Predicting'):
        orig_img       = np.array(Image.open(os.path.join(val_dir, img_file)).convert('RGB'))
        orig_h, orig_w = orig_img.shape[:2]

        aug          = val_tf(image=orig_img, bboxes=[], class_labels=[])
        image_tensor = aug['image'].unsqueeze(0).to(DEVICE)

        if use_tta:
            output = tta_predict(model, image_tensor)
        else:
            with torch.no_grad():
                out    = model(image_tensor)[0]
                output = {k: v.cpu() for k, v in out.items()}

        # Scale boxes back to original resolution
        inp_h, inp_w = image_tensor.shape[-2:]
        scale_x = orig_w / inp_w
        scale_y = orig_h / inp_h
        boxes  = output['boxes'].numpy().copy()
        scores = output['scores'].numpy()
        labels = output['labels'].numpy()
        if len(boxes):
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y

        # ── Visualise ────────────────────────────────────────────────────
        fig, ax = plt.subplots(1, figsize=(5, 5))
        ax.imshow(orig_img); ax.axis('off')
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = map(int, box.tolist())
            lname = REV_LABEL_MAP.get(int(label), 'unknown')
            rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=1.5, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, max(y1-4, 0), f'{lname} {score:.2f}', color='red',
                    fontsize=7, weight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        fig.tight_layout(pad=0)
        fig.savefig(os.path.join(save_img_dir, img_file), bbox_inches='tight', dpi=100)
        plt.close(fig)

        # ── Build ONE row per image, space-separated multi-detection ─────
        if len(boxes):
            # Sort by score descending — highest confidence first
            order  = np.argsort(scores)[::-1]
            boxes  = boxes[order]
            scores = scores[order]
            labels = labels[order]

            best_label = REV_LABEL_MAP.get(int(labels[0]), 'unknown')

            # Round cf to 2 decimal places (matches sample: 0.10, 0.09 0.05)
            cf_str   = ' '.join(f'{s:.2f}'     for s in scores)
            xmin_str = ' '.join(str(int(b[0])) for b in boxes)
            ymin_str = ' '.join(str(int(b[1])) for b in boxes)
            xmax_str = ' '.join(str(int(b[2])) for b in boxes)
            ymax_str = ' '.join(str(int(b[3])) for b in boxes)

            rows.append({
                'ID':    img_file,
                'label': best_label,
                'cf':    cf_str,
                'xmin':  xmin_str,
                'ymin':  ymin_str,
                'xmax':  xmax_str,
                'ymax':  ymax_str,
            })
        else:
            # No detection — matches sample format: none, 0, 0, 0, 0, 0
            rows.append({
                'ID':    img_file,
                'label': 'none',
                'cf':    '0',
                'xmin':  '0',
                'ymin':  '0',
                'xmax':  '0',
                'ymax':  '0',
            })

    df = pd.DataFrame(rows, columns=['ID', 'label', 'cf', 'xmin', 'ymin', 'xmax', 'ymax'])
    df.to_csv(save_csv_path, index=False)
    n_detected = (df['label'] != 'none').sum()
    print(f'\n✓ submission.csv  → {save_csv_path}')
    print(f'  {len(df)} rows total | {n_detected} with detections | {len(df)-n_detected} none')
    print(f'✓ viz images      → {save_img_dir}/')
    return df

# 10. Build data loaders

In [13]:
train_dataset = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT,
                               transforms=get_train_transforms(),
                               use_mosaic=True)   # 30% mosaic
sampler       = get_balanced_sampler(train_dataset)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                            collate_fn=collate_fn, num_workers=2, pin_memory=True)

val_size    = min(200, int(0.1 * len(train_dataset)))
val_indices = list(range(len(train_dataset) - val_size, len(train_dataset)))
val_dataset = torch.utils.data.Subset(
    NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT,
                  transforms=get_val_transforms(), use_mosaic=False),
    val_indices)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'train batches: {len(train_loader)} | val batches: {len(val_loader)}')
print(f'Effective batch size: {BATCH_SIZE} × {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}')

/tmp/ipykernel_10964/1102527643.py:4: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_10964/1102527643.py:16: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 60)),
/tmp/ipykernel_10964/1102527643.py:27: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=40, max_width=40, fill_value=0, p=0.3),
/tmp/ipykernel_10964/1102527643.py:37: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),


train batches: 360 | val batches: 72
Effective batch size: 4 × 3 = 12


#  11. Train or load, then infer

In [ ]:

if not INFER_ONLY:
    model = train_model(model, train_loader, val_loader, epochs=EPOCHS)
else:
    wp    = EMA_PATH if os.path.exists(EMA_PATH) else CHECKPOINT_PATH
    state = torch.load(wp, map_location='cpu')
    model.load_state_dict(state['model'] if 'model' in state else state)
    model = model.to(DEVICE)
    print(f'Loaded weights from {wp}')

submission_df = predict_and_export(
    model, VAL_IMG_ROOT, SUBMISSION_PATH,
    save_img_dir=VIZ_DIR, use_tta=USE_TTA)

submission_df.head(10)

Epoch   1/2:   0%|          | 0/360 [00:00<?, ?it/s]/tmp/ipykernel_10964/1102527643.py:53: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
/tmp/ipykernel_10964/1102527643.py:53: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
Epoch   1/2:  11%|█▏        | 41/360 [01:44<12:01,  2.26s/it, loss=3.504, lr=1.1e-06]

# 12. Sanity checks

In [ ]:
print('Label distribution in submission:')
print(submission_df['label'].value_counts())
print(f'\nTotal rows   : {len(submission_df)}')
print(f'Unique images: {submission_df["ID"].nunique()}')
print('\nSample rows:')
print(submission_df.head(10).to_string(index=False))

imgs = sorted(os.listdir(VIZ_DIR))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, fname in zip(axes.flatten(), imgs):
    ax.imshow(Image.open(os.path.join(VIZ_DIR, fname)))
    ax.set_title(fname.split('.')[0], fontsize=7); ax.axis('off')
for ax in axes.flatten()[len(imgs):]: ax.axis('off')
plt.suptitle('Sample predictions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
import os

paths = {
    "Submission CSV": SUBMISSION_PATH,
    "Train History CSV": OUTPUT_DIR + '/train_history.csv',
    "Best Checkpoint": CHECKPOINT_PATH,
    "EMA Model": EMA_PATH,
    "Viz Images Dir": VIZ_DIR,
}

print("=" * 50)
print("OUTPUT FILE LOCATIONS")
print("=" * 50)
for name, path in paths.items():
    exists = os.path.exists(path)
    status = "✅ EXISTS" if exists else "❌ NOT FOUND"
    size = f"  ({os.path.getsize(path) / 1024:.1f} KB)" if exists and os.path.isfile(path) else ""
    print(f"{status}  |  {name}")
    print(f"          →  {path}{size}")
print("=" * 50)